In [9]:
import pandas as pd
import requests
from datetime import datetime

# ====== 配置区 ======
# 如果没有 NY Open Data 的 app token，就留空字符串，不会报错
APP_TOKEN = ""   # e.g. "xxxxxxxxxxxx"

# 你现在用的 MTA Subway Hourly Ridership 数据集
API_URL = "https://data.ny.gov/resource/5wq4-mkjj.json"

# 车站 → ZIP 静态映射（可以后面再慢慢补充）
STATION_TO_ZIP_MAP = {
    'Flushing-Main St (7)': '11354',
    'Mets-Willets Point (7)': '11354',

    'Astor Pl (6)': '10003',
    '8 St-NYU (N,R,W)': '10003',
    '14 St-Union Sq (L,N,Q,R,W,4,5,6)': '10003',
    '3 Av (L)': '10003',

    '34 St-Penn Station (A,C,E)': '10001',
    '34 St-Penn Station (1,2,3)': '10001',
    '34 St-Herald Sq (B,D,F,M,N,Q,R,W)': '10001',
    '28 St (1)': '10001',
    '28 St (N,R,W)': '10001',

    '116 St-Columbia University (1)': '10027',
    '125 St (1)': '10027',
    '125 St (A,B,C,D)': '10027',

    'Vernon Blvd-Jackson Av (7)': '11101',
    'Hunters Point Av (7)': '11101',
    'Court Sq (7)': '11101',
    'Queensboro Plaza (7,N,W)': '11101',

    'Jay St-MetroTech (A,C,F,R)': '11201',
    'Borough Hall (2,3,4,5)': '11201',

    'Forest Hills-71 Av (E,F,M,R)': '11375',
    '67 Av (M,R)': '11375',

    '4 Av-9 St (F,G,R)': '11215',
    '7 Av (F,G)': '11215',

    'Fordham Rd (4)': '10458',
    'Fordham Rd (B,D)': '10458',

    'St George (SIR)': '10314'
}

In [10]:
def fetch_and_clean_mta_data(start_date='2025-01-01'):
    """
    从 NYC Open Data API 拉取 MTA Subway Hourly Ridership，
    映射到 ZIP，并按月汇总。

    返回 DataFrame：
    columns = [zip_code, report_month, total_foot_traffic]
    """
    print("🚀 [MTA Pipeline] Starting execution...")

    # ===== 1. Extract: 从 API 拉数据 =====
    where_clause = f"transit_timestamp >= '{start_date}T00:00:00'"
    params = {
        "$select": "transit_timestamp,station_complex,ridership",
        "$where": where_clause,
        "$order": "transit_timestamp DESC",
        "$limit": 3000000   # 防止一次拿太少
    }

    headers = {}
    if APP_TOKEN:  # 如果你以后加了 token 就会自动带上
        headers["X-App-Token"] = APP_TOKEN

    try:
        print(f"📡 Fetching data from {API_URL} ...")
        response = requests.get(API_URL, params=params, headers=headers)
        response.raise_for_status()
        raw_data = response.json()
    except Exception as e:
        print(f"❌ API request failed: {e}")
        return pd.DataFrame()

    df = pd.DataFrame(raw_data)
    print(f"✅ Fetched {len(df)} raw rows.")

    if df.empty:
        print("⚠️ No data returned, exiting.")
        return pd.DataFrame()

    # ===== 2. Transform: 类型转换 + 映射 ZIP =====
    # 时间
    df['transit_timestamp'] = pd.to_datetime(df['transit_timestamp'])
    # 人流量
    df['ridership'] = pd.to_numeric(df['ridership'], errors='coerce').fillna(0)

    # 车站 → ZIP 静态字典映射
    df['zip_code'] = df['station_complex'].map(STATION_TO_ZIP_MAP)
    df_filtered = df.dropna(subset=['zip_code']).copy()
    print(f"📍 Rows with target ZIPs: {len(df_filtered)}")

    if df_filtered.empty:
        print("⚠️ No rows matched target ZIPs, check STATION_TO_ZIP_MAP.")
        return pd.DataFrame()

    # ===== 3. Aggregate: 按 ZIP + 月 聚合 =====
    print("∑ Aggregating MONTHLY traffic per ZIP ...")
    df_monthly = (
        df_filtered
        .groupby(
            ['zip_code',
             pd.Grouper(key='transit_timestamp', freq='MS')]  # Month Start
        )['ridership']
        .sum()
        .reset_index()
    )

    # 重命名列
    df_monthly.rename(columns={
        'transit_timestamp': 'report_month',
        'ridership': 'total_foot_traffic'
    }, inplace=True)

    # 格式化类型
    df_monthly['report_month'] = df_monthly['report_month'].dt.date
    df_monthly['zip_code'] = df_monthly['zip_code'].astype(str)
    df_monthly['total_foot_traffic'] = df_monthly['total_foot_traffic'].astype(int)

    print(f"✅ [MTA Pipeline] Ready. Rows: {len(df_monthly)}")
    return df_monthly

In [12]:
# 调用函数，生成最终表
df_mta = fetch_and_clean_mta_data()

# 看前几行
df_mta.head()

🚀 [MTA Pipeline] Starting execution...
📡 Fetching data from https://data.ny.gov/resource/5wq4-mkjj.json ...
✅ Fetched 3000000 raw rows.
📍 Rows with target ZIPs: 164163
∑ Aggregating MONTHLY traffic per ZIP ...
✅ [MTA Pipeline] Ready. Rows: 20


,zip_code,report_month,total_foot_traffic
0,10001,2025-10-01,4738815
1,10001,2025-11-01,2635693
2,10003,2025-10-01,2224871
3,10003,2025-11-01,1210286
4,10027,2025-10-01,469145


In [13]:
df_mta.to_csv("mta_monthly_traffic.csv", index=False)